# Stream 2 Abdominal Circumference (AC) Landmark Model Evaluation

## 1. Purpose & Protocol
This notebook documents and verifies the official test set evaluation of the Stream 2 Abdominal Circumference landmark localization models.

Key aspects:
- **Test Set**: 147 official test images (114 unique patients) from fetal ultrasound datasets.
- **Architectures**:
  - **Single Model (`v3_full`)**: HRNet/ResNet landmark localization predicting 4 anatomical landmarks with horizontal flip-TTA and robust ellipse diameter calculation.
  - **4-Model Ensemble (`ac_v3_ens`)**: Ensemble of four distinct seed models (s1, s2, s3, s4) aggregated via median coordinate prediction with flip-TTA.
- **Published Target Baselines**:
  - `v3_full` AC MAE: **4.395 mm** (reproduced: 4.3951 mm).
  - 4-Model Ensemble Median AC MAE: **4.343 mm** (reproduced: 4.3426 mm).


In [1]:
import sys
from pathlib import Path
import subprocess

work_dir = Path.cwd()
while work_dir.name != "work" and work_dir.parent != work_dir:
    work_dir = work_dir.parent
if str(work_dir / "src") not in sys.path:
    sys.path.insert(0, str(work_dir / "src"))

print(f"Work directory: {work_dir}")
script_path = work_dir / "scripts" / "06_eval_ac.py"
print(f"Executing {script_path.name} on CPU...")

proc = subprocess.run(
    [sys.executable, str(script_path)],
    cwd=str(work_dir),
    capture_output=True,
    text=True,
    check=True,
)
print(proc.stdout)


Work directory: C:\fetal-github-clone\work
Executing 06_eval_ac.py on CPU...


06_eval_ac: Evaluating AC checkpoints on official test set
  Device   : cpu
  Data dir : C:/fetal-data

[Data] Official test set loaded: 147 images (114 patients)

--- 1. Evaluating Single Model: v3_full ---
Checkpoint: C:/fetal-models/ac_v3/v3_full/best.pt

[RESULT: v3_full (flip-TTA)]
  AC MAE       : 4.3951 mm  (Baseline: 4.3951 mm)
  AC RMSE      : 6.1540 mm  (Baseline: 6.1540 mm)
  AC Median    : 2.9008 mm
  AC Bias      : -1.5368 mm
  TAD MAE      : 1.5046 mm
  APAD MAE     : 1.7617 mm
  NME          : 0.0731
  >10mm Outlier: 8.16%

--- 2. Evaluating 4-Model Ensemble: s1-s4 (Median Aggregation) ---
  Seed ckpt: C:/fetal-models/ac_v3_ens/s1/best.pt
  Seed ckpt: C:/fetal-models/ac_v3_ens/s2/best.pt
  Seed ckpt: C:/fetal-models/ac_v3_ens/s3/best.pt
  Seed ckpt: C:/fetal-models/ac_v3_ens/s4/best.pt

[RESULT: 4-Model Ensemble Median (flip-TTA)]
  AC MAE       : 4.3426 mm  (Baseline: 4.3426 mm)
  AC RMSE      : 6.0305 mm  (Baseline: 6.0305 mm)
  AC Median    : 3.1467 mm
  AC Bias      

In [2]:
import json
from pathlib import Path
import pandas as pd

ac_dir = work_dir / "docs" / "results" / "ac"
v3_json = ac_dir / "ac_v3" / "v3_full" / "results.json"
ens_json = ac_dir / "ac_v3_ens" / "results.json"

with open(v3_json, "r", encoding="utf-8") as f:
    v3_data = json.load(f)
with open(ens_json, "r", encoding="utf-8") as f:
    ens_data = json.load(f)

v3_metrics = v3_data.get("test_tta", v3_data.get("test", v3_data))

print("=== AC Model & Seed Ensemble Performance (147 Official Test Images) ===\n")
rows = [
    {
        "Model / Configuration": "Single Model: v3_full (TTA)",
        "AC MAE (mm)": f"{v3_metrics['ac_mae']:.4f}",
        "AC RMSE (mm)": f"{v3_metrics['ac_rmse']:.4f}",
        "AC Median (mm)": f"{v3_metrics['ac_median']:.4f}",
        "TAD MAE (mm)": f"{v3_metrics['tad_mae']:.4f}",
        "APAD MAE (mm)": f"{v3_metrics['apad_mae']:.4f}",
        "NME": f"{v3_metrics['nme']:.4f}",
        "Outliers >10mm": f"{v3_metrics['over10']*100:.2f}%",
    }
]

for s in ["s1", "s2", "s3", "s4"]:
    sd = ens_data[s]
    rows.append({
        "Model / Configuration": f"Seed Model: {s}",
        "AC MAE (mm)": f"{sd['ac_mae']:.4f}",
        "AC RMSE (mm)": f"{sd['ac_rmse']:.4f}",
        "AC Median (mm)": f"{sd['ac_median']:.4f}",
        "TAD MAE (mm)": f"{sd['tad_mae']:.4f}",
        "APAD MAE (mm)": f"{sd['apad_mae']:.4f}",
        "NME": f"{sd['nme']:.4f}",
        "Outliers >10mm": f"{sd['over10']*100:.2f}%",
    })

for ens_type in ["ENS_mean", "ENS_median"]:
    ed = ens_data[ens_type]
    rows.append({
        "Model / Configuration": f"Ensemble: {ens_type}",
        "AC MAE (mm)": f"{ed['ac_mae']:.4f}",
        "AC RMSE (mm)": f"{ed['ac_rmse']:.4f}",
        "AC Median (mm)": f"{ed['ac_median']:.4f}",
        "TAD MAE (mm)": f"{ed['tad_mae']:.4f}",
        "APAD MAE (mm)": f"{ed['apad_mae']:.4f}",
        "NME": f"{ed['nme']:.4f}",
        "Outliers >10mm": f"{ed['over10']*100:.2f}%",
    })

df_seeds = pd.DataFrame(rows)
print(df_seeds.to_string(index=False))

print("\n=== Verification Check Against Published Targets ===")
diff_v3 = abs(v3_metrics["ac_mae"] - 4.3951)
diff_ens = abs(ens_data["ENS_median"]["ac_mae"] - 4.3426)
print(f"  Single Model (v3_full)  AC MAE: {v3_metrics['ac_mae']:.4f} mm  (Published: 4.395 mm, diff: {diff_v3:.6f} mm)")
print(f"  Ensemble Median (s1-s4) AC MAE: {ens_data['ENS_median']['ac_mae']:.4f} mm  (Published: 4.343 mm, diff: {diff_ens:.6f} mm)")
print(f"  v3_full check pass (<= 0.002 mm): {diff_v3 <= 0.002}")
print(f"  Ensemble check pass (<= 0.002 mm): {diff_ens <= 0.002}")


=== AC Model & Seed Ensemble Performance (147 Official Test Images) ===

      Model / Configuration AC MAE (mm) AC RMSE (mm) AC Median (mm) TAD MAE (mm) APAD MAE (mm)    NME Outliers >10mm
Single Model: v3_full (TTA)      4.3956       6.1573         2.9078       1.5038        1.7626 0.0731          8.16%
             Seed Model: s1      4.9622       7.3273         3.1394       1.7208        2.2514 0.0796         12.24%
             Seed Model: s2      4.3220       6.0468         2.7327       1.5945        1.6803 0.0770          8.84%
             Seed Model: s3      4.5439       6.5892         3.3688       1.7229        1.8244 0.0789         10.20%
             Seed Model: s4      4.4916       6.2692         3.6400       1.5366        1.8217 0.0765          8.16%
         Ensemble: ENS_mean      4.4693       6.3088         3.0115       1.5994        1.7223 0.0751          9.52%
       Ensemble: ENS_median      4.3430       6.0343         3.1235       1.6001        1.7034 0.0757       